# 📊 Business Aggregated Profit Pipeline

This pipeline reads curated order data from Delta tables, performs joins and transformations, and writes an aggregated profit table for reporting and analysis.

---

## ✅ Objective

To generate an aggregated profit table grouped by:
- Year
- Product Category
- Sub-Category
- Customer Name

---



In [0]:
from pyspark.sql import functions as F

# ---------------------------
# Read curated tables
# ---------------------------
order_details = spark.table("curated_order_details")
order_header = spark.table("curated_order_header")
customers = spark.table("curated_customer_master")

# ---------------------------
# Join order details + order header on order_id
# then join customer on customer_id from order_header
# ---------------------------
orders_full = (
    order_details
    .join(order_header, on="order_id", how="left")
    .join(customers.select("customer_id", "customer_name"), on="customer_id", how="left")
)

# ---------------------------
# Extract year from order_date (from order_header)
# ---------------------------
orders_full = orders_full.withColumn(
    "year", F.year(F.to_date("order_date", "d/M/yyyy"))
)

# ---------------------------
# Aggregate by Year, Product Category, Sub Category, Customer
# ---------------------------
business_agg_profit = orders_full.groupBy(
    "year", "category", "sub_category", "customer_name"
).agg(
    F.round(F.sum("profit"), 2).alias("total_profit")
)

# ---------------------------
# Save as Delta table
# ---------------------------
business_agg_profit.write.format("delta").mode("overwrite").saveAsTable("business_agg_profit")
print("✅ Aggregated profit table created: business_agg_profit")

# Optional: show top rows
business_agg_profit.show(10)


# 🧮 Temporary Views for Profit Analysis

This section creates **temporary views** in SQL for slicing the `business_agg_profit` data across various business dimensions like year, category, and customer.

These views can be queried directly for dashboards or further data analysis.

---


In [0]:
%sql
-- Profit by Year
CREATE OR REPLACE TEMP VIEW temp_profit_by_year AS
SELECT 
    year,
    ROUND(SUM(total_profit), 2) AS profit_by_year
FROM business_agg_profit
GROUP BY year
ORDER BY year;

-- Profit by Year + Product Category
CREATE OR REPLACE TEMP VIEW temp_profit_by_year_category AS
SELECT 
    year,
    category AS product_category,
    ROUND(SUM(total_profit), 2) AS profit_by_year_category
FROM business_agg_profit
GROUP BY year, category
ORDER BY year, product_category;

-- Profit by Customer
CREATE OR REPLACE TEMP VIEW temp_profit_by_customer AS
SELECT 
    customer_name,
    ROUND(SUM(total_profit), 2) AS profit_by_customer
FROM business_agg_profit
GROUP BY customer_name
ORDER BY profit_by_customer DESC;

-- Profit by Customer + Year
CREATE OR REPLACE TEMP VIEW temp_profit_by_customer_year AS
SELECT 
    customer_name,
    year,
    ROUND(SUM(total_profit), 2) AS profit_by_customer_year
FROM business_agg_profit
GROUP BY customer_name, year
ORDER BY customer_name, year;
